# Multiscale Equilibria and Jacobian Regularization

A multiscale equilibrium solves several coupled states at once. In a compact
two-scale notation,

$$
z^\star=(z_\ell^\star,z_h^\star),
\qquad
z^\star=f_\theta(z^\star,x).
$$

The tutorial block uses

$$
z_\ell^+=\tanh(S_\ell(x)+A_{\ell\ell}z_\ell+A_{h\ell}z_h),
$$

$$
z_h^+=\tanh(S_h(x)+A_{hh}z_h+A_{\ell h}z_\ell).
$$

Jacobian regularization adds a penalty such as

$$
\lambda_J\|J_f(z^\star)\|_F^2
$$

to encourage stable local dynamics.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import resolve_device, SolverConfig

torch.manual_seed(7)
np.random.seed(7)
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
device

## Two Coupled Scales

The state is stored as one tensor for solver compatibility, then split into
low and high scales when the transition is evaluated.

In [ ]:
from silva_networks import (
    silva_jacobian_regularization_loss,
    silva_multiscale_deq_block,
    stability_report,
)

x = torch.randn(6, 4, device=device)
block = silva_multiscale_deq_block(
    in_dim=4,
    low_dim=3,
    high_dim=5,
    config=SolverConfig(solver="anderson", max_iter=8, alpha=0.6, history=3),
).to(device)

result = block(x, return_result=True)
z_low, z_high = block.split_state(result.z)
result.z.shape, z_low.shape, z_high.shape, result.residual

## Hutchinson Estimate

For a random Rademacher vector $v$,

$$
\mathbb E\|J_f^\top v\|_2^2=\|J_f\|_F^2.
$$

The estimator uses VJP calls and does not materialize the full Jacobian.

In [ ]:
penalty = silva_jacobian_regularization_loss(
    lambda z: block.transition(z, x),
    result.z,
    samples=2,
    squared=True,
    weight=0.01,
)
penalty.detach().cpu()

## Stability Report

The residual and spectral radius summarize the fixed point locally:

$$
\rho(J_f(z^\star))<1
$$

is the familiar contraction-style target for stable Picard dynamics.

In [ ]:
report = stability_report(lambda z: block.transition(z, x), result.z, samples=1, iters=6)
report

## Training Loss with a Jacobian Penalty

This cell separates the task loss from the regularizer so the effect of the
penalty is visible in the output.

In [ ]:
head = torch.nn.Linear(block.state_dim, 2).to(device)
y = torch.randint(0, 2, (x.shape[0],), device=device)
optim = torch.optim.Adam(list(block.parameters()) + list(head.parameters()), lr=0.01)

losses = []
penalties = []
for _ in range(4):
    optim.zero_grad()
    solve = block(x, return_result=True)
    logits = head(solve.z)
    task_loss = torch.nn.functional.cross_entropy(logits, y)
    jac_loss = silva_jacobian_regularization_loss(lambda z: block.transition(z, x), solve.z, samples=1, weight=0.01)
    loss = task_loss + jac_loss
    loss.backward()
    optim.step()
    losses.append(float(task_loss.detach().cpu()))
    penalties.append(float(jac_loss.detach().cpu()))

losses, penalties

In [ ]:
plt.figure(figsize=(5.2, 3.0))
plt.plot(losses, marker="o", label="task loss")
plt.plot(penalties, marker="s", label="Jacobian penalty")
plt.xlabel("training step")
plt.legend()
plt.tight_layout()

## Citation and Sources

If this package or notebook is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA methodology, cite the SILVA Networks
paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background sources:

- Deep Implicit Layers tutorial: https://implicit-layers-tutorial.org/
- LocusLab DEQ repository: https://github.com/locuslab/deq
- Deep Equilibrium Models: https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models: https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization: https://arxiv.org/abs/2106.14342

The notebook is adapted to the `silva_networks` public API. It links to the
sources above and keep the examples package-native.